# TP 1: LDA/QDA y optimización matemática de modelos

Se puede consultar la introducción teórica en el mono-notebook, se prefiere mantener este lo más chico posible.

In [1]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)


## Ejemplo

In [2]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [3]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [4]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [5]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [6]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.429063,2.848945,2.643208,1.391843,0.982407,0.018784,0.000670,0.008000,0.000437
TensorizedQDA,0.413854,5.023431,1.276520,0.657844,0.982593,0.018784,0.000653,0.011948,0.000046


In [8]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.429063,2.643208,0.982407
TensorizedQDA,0.413854,1.276520,0.982593


In [9]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.429063,2.848945,2.643208,1.391843,0.982407,0.018784,0.000670,0.008000,0.000437,1.000000,1.000000,1.0,1.00000
TensorizedQDA,0.413854,5.023431,1.276520,0.657844,0.982593,0.018784,0.000653,0.011948,0.000046,1.036748,2.070635,1.0,0.66962


In [10]:
# volvemos a subsetear columnas
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.429063,2.643208,0.982407,1.000000,1.000000,1.0,1.00000
TensorizedQDA,0.413854,1.276520,0.982593,1.036748,2.070635,1.0,0.66962


### Diferencias entre `QDA`y `TensorizedQDA`

#### 1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

**QDA** no sobreescribe `_predict_one`, por lo que usa la misma implementación que la clase padre `BaseBayesianClassifier`:

In [ ]:
  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

Aquí la función recibe un parámetro `x` para el cual construye `log_posteriori` iterando sobre `k` classes, cada una de las cuales llama a `_predict_log_conditional(x, idx)`. Pero además, el trigger está en el método `predict`, donde se itera sobre cantidad de observaciones (`m_obs`):

In [ ]:
def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

O sea que es un doble loop, o nested loop `n->k`, lo que equivale a `n*k` computations.

**TensorizedQDA** también usa el método `predict` de `BaseBayesianClassifier`, pero sobreescribe `_predict_one`:

In [ ]:
def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

Esto le permite reemplazar el `k-loop` con un único llamado a `_predict_log_conditionals` (que usa `tensor_inv_cov / tensor_means`). O sea que itera (o paraleliza, si spawneamos procesos paralelos) sobre `n` observaciones únicamente.

#### 2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

In [11]:
# Shapes y chequeo numérico vs QDA (misma separación train/test que el benchmark)
from utils.datasets import split_transpose

X_train, X_test, y_train, y_test = split_transpose(
    X_full, y_full_encoded, test_size=0.3, random_state=6553
)

tqda = TensorizedQDA()
tqda.fit(X_train, y_train)

p, n_train = X_train.shape
k = len(tqda.log_a_priori)

print("p (features), n_train:", p, n_train)
print("k (clases):", k)
print("tensor_inv_cov.shape:", tqda.tensor_inv_cov.shape, "  # (k, p, p)")
print("tensor_means.shape:", tqda.tensor_means.shape, "  # (k, p, 1)")

x = X_test[:, :1]
print("x (un punto test).shape:", x.shape, "  # (p, 1)")

u = x - tqda.tensor_means
print("unbiased_x = x - tensor_means ->", u.shape, "  # (k, p, 1)")

ut = u.transpose(0, 2, 1)
print("unbiased_x.transpose(0,2,1) ->", ut.shape, "  # (k, 1, p)")

inner = ut @ tqda.tensor_inv_cov @ u
print("row @ inv_cov @ col (batched) ->", inner.shape, "  # (k, 1, 1)")

qda = QDA()
qda.fit(X_train, y_train)

log_t = tqda._predict_log_conditionals(x)
log_loop = np.array([qda._predict_log_conditional(x, j) for j in range(k)])
print("allclose(log condicionales):", np.allclose(log_t, log_loop))
print("_predict_one coincide:", qda._predict_one(x) == tqda._predict_one(x))

p (features), n_train: 13 124
k (clases): 3
tensor_inv_cov.shape: (3, 13, 13)   # (k, p, p)
tensor_means.shape: (3, 13, 1)   # (k, p, 1)
x (un punto test).shape: (13, 1)   # (p, 1)
unbiased_x = x - tensor_means -> (3, 13, 1)   # (k, p, 1)
unbiased_x.transpose(0,2,1) -> (3, 1, 13)   # (k, 1, p)
row @ inv_cov @ col (batched) -> (3, 1, 1)   # (k, 1, 1)
allclose(log condicionales): False
_predict_one coincide: True


| Pasos | QDA | TensorizedQDA |
|:--|:--|:--|
| **1. Parámetros** | Listas `inv_covs[c]` con shape `(p,p)` y `means[c]` con shape `(p,1)` para `c=0,ldots,k-1`. | `tensor_inv_cov` con shape `(k,p,p)` y `tensor_means` con shape `(k,p,1)`. El elemento en el índice `c` coincide con `inv_covs[c]` y `means[c]`. |
| **2. Entrada** | Columna `x` con shape `(p,1)` (lo arma `predict` para cada punto). | El mismo `x` con shape `(p,1)`. |
| **3. Centrar** | Para la clase `c`, dentro de `_predict_log_conditional(x, c)`: `x - means[c]` con shape `(p,1)`. | `x - tensor_means` con broadcasting y shape `(k,p,1)`. La fila `c` es lo mismo que `x - means[c]`. |
| **4. Forma cuadrática** | Para la clase `c`: `unbiased.T @ inv_covs[c] @ unbiased`, un escalar. | `unbiased_x.transpose(0,2,1) @ tensor_inv_cov @ unbiased_x` con shapes `(k,1,p) @ (k,p,p) @ (k,p,1)` => `(k,1,1)`. |
| **5. Término log-det** | En cada llamada por clase: `0.5 * log(det(inv_covs[c]))`. | `0.5 * log(det(tensor_inv_cov))`: `det` sobre cada bloque `(p,p)` resulta en un vector de k valores. |
| **6. log p(x\mid c)** | `_predict_log_conditional` devuelve un escalar; `_predict_one` junta k valores en una lista. | `_predict_log_conditionals` devuelve un array de longitud k. |
| **7. Predicción** | `argmax` sobre `log_a_priori[c] + log p(x\mid c)` para cada c. | `argmax` sobre `log_a_priori +` el vector del paso 6 (suma broadcast). Misma clase ganadora. |

```mermaid
flowchart LR
  x["x (p,1)"]
  bm["tensor_means (k,p,1)"]
  u["x - means (k,p,1)"]
  mat["batched (k,1,p)@(k,p,p)@(k,p,1)"]
  v["k log-condicionales"]
  am["argmax"]
  x --> u
  bm --> u
  u --> mat
  mat --> v
  v --> am
```

Los shapes concretos para un split de entrenamiento se imprimen en la celda siguiente.

### Optimización

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.

In [ ]:
class FasterQDA(TensorizedQDA):

    # override `predict`: en vez de iterar sobre cada observación, calcula todo el producto matricial de una vez (sin for-loop)
    def predict(self, X):
        # X shape: (p, n)

        # tensor_means shape: (k, p, 1)
        # unbiased_X shape: (k, p, n) — extraemos el mean de cada clase y se lo restamos a cada observación (broadcasting)
        unbiased_X = X - self.tensor_means  # broadcasts (p,n) - (k,p,1) -> (k,p,n)

        # tensor_inv_cov shape: (k, p, p)
        # producto matricial entre la matriz de covarianza y el vector de cada observación (obs - tensor_mean)
        # (k,p,p) @ (k,p,n) -> (k,p,n)
        AX = self.tensor_inv_cov @ unbiased_X

        # devuelve la diagonal en formato -> (k, n)
        quad_forms = np.sum(AX * unbiased_X, axis=1)

        # log determinants: (k,)  -> reshape a (k,1) para broadcasting
        log_dets = 0.5 * np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)

        # log conditionals shape: (k, n)
        log_conditionals = log_dets - 0.5 * quad_forms

        # log posteriori shape: (k, n)
        log_posteriori = self.log_a_priori.reshape(-1, 1) + log_conditionals

        # return shape (1, n)
        return np.argmax(log_posteriori, axis=0).reshape(1, -1)

In [ ]:
# testing
qda = QDA()
fqda = FasterQDA()

qda.fit(X_train, y_train)
fqda.fit(X_train, y_train)

pred_qda = qda.predict(X_test)
pred_fqda = fqda.predict(X_test)

print(np.array_equal(pred_qda, pred_fqda))

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.

In [ ]:
class BaseBayesianClassifier:

    # ... otros métodos ...
    def predict(self, X):
        m_obs = X.shape[1]
        y_hat = np.empty(m_obs, dtype=int)

        # acá estamos iterando sobre cada observación, haciendo una predicción individual para cada una, y guardando el resultado en y_hat
        for i in range(m_obs):
            y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

        # ...